# Install the library (run this cell if using Colab or if you haven't installed the package)
!pip install --upgrade --no-cache-dir simple-backtest yfinance

In [ ]:
# Install the library (run this cell if using Colab or if you haven't installed the package)
!pip install simple-backtest yfinance

In [ ]:
import pandas as pd
import yfinance as yf

from simple_backtest import Backtest, BacktestConfig
from simple_backtest.commission import (
    Commission,
)
from simple_backtest.strategy import MovingAverageStrategy

## Load Data

In [ ]:
# Download data
ticker = "SPY"
data = yf.download(ticker, start="2020-01-01", end="2023-12-31", progress=False)
data = data.dropna()

print(f"Data shape: {data.shape}")
print(f"Date range: {data.index[0]} to {data.index[-1]}")

# Download data
ticker = "SPY"
data = yf.download(ticker, start="2020-01-01", end="2023-12-31", progress=False)

# Handle MultiIndex columns if present
if isinstance(data.columns, pd.MultiIndex):
    data.columns = data.columns.get_level_values(0)

data = data.dropna()

print(f"Data shape: {data.shape}")
print(f"Date range: {data.index[0]} to {data.index[-1]}")

In [ ]:
# Test percentage commission
config_percentage = BacktestConfig(
    initial_capital=10000.0,
    lookback_period=50,
    commission_type="percentage",
    commission_value=0.001,  # 0.1% per trade
    risk_free_rate=0.02,
)

print("Percentage Commission Configuration:")
print(f"  Commission: {config_percentage.commission_value * 100}% per trade")
print("\nExamples:")
print(f"  Trade $1,000:  Commission = ${1000 * 0.001:.2f}")
print(f"  Trade $5,000:  Commission = ${5000 * 0.001:.2f}")
print(f"  Trade $10,000: Commission = ${10000 * 0.001:.2f}")

In [ ]:
# Run backtest with percentage commission
strategy = MovingAverageStrategy(short_window=10, long_window=30, shares=10)

backtest = Backtest(data=data, config=config_percentage)
results_percentage = backtest.run([strategy])

metrics = results_percentage[strategy.get_name()]["metrics"]
print(f"\nPerformance with {config_percentage.commission_value * 100}% Commission:")
print(f"  Total Return:  {metrics['total_return']:.2f}%")
print(f"  Total Trades:  {metrics['total_trades']}")
print(f"  Sharpe Ratio:  {metrics['sharpe_ratio']:.2f}")

## 2. Flat Commission

**Fixed fee per trade**, regardless of size.

Examples:
- Robinhood: $0 (free trading)
- Traditional brokers: $5-$10 per trade
- Some forex brokers: $0-$5

In [ ]:
# Test flat commission
config_flat = BacktestConfig(
    initial_capital=10000.0,
    lookback_period=50,
    commission_type="flat",
    commission_value=5.0,  # $5 per trade
    risk_free_rate=0.02,
)

print("Flat Commission Configuration:")
print(f"  Commission: ${config_flat.commission_value:.2f} per trade")
print("\nExamples:")
print(f"  Trade $1,000:  Commission = ${config_flat.commission_value:.2f}")
print(f"  Trade $5,000:  Commission = ${config_flat.commission_value:.2f}")
print(f"  Trade $10,000: Commission = ${config_flat.commission_value:.2f}")

In [ ]:
# Run backtest with flat commission
backtest = Backtest(data=data, config=config_flat)
results_flat = backtest.run([strategy])

metrics = results_flat[strategy.get_name()]["metrics"]
print(f"\nPerformance with ${config_flat.commission_value:.2f} Flat Commission:")
print(f"  Total Return:  {metrics['total_return']:.2f}%")
print(f"  Total Trades:  {metrics['total_trades']}")
print(f"  Sharpe Ratio:  {metrics['sharpe_ratio']:.2f}")

## 3. Tiered Commission

**Volume-based pricing**: Larger trades get lower commission rates.

Common for:
- Institutional brokers
- High-volume traders
- Some crypto exchanges

In [ ]:
# Test tiered commission
# Define tiers: (threshold, rate)
tiers = [
    (1000, 0.002),  # $0-$1,000: 0.2%
    (5000, 0.001),  # $1,000-$5,000: 0.1%
    (float("inf"), 0.0005),  # $5,000+: 0.05%
]

config_tiered = BacktestConfig(
    initial_capital=10000.0,
    lookback_period=50,
    commission_type="tiered",
    commission_value=tiers,
    risk_free_rate=0.02,
)

print("Tiered Commission Configuration:")
print("  Tiers:")
print("    $0 - $1,000:    0.2% commission")
print("    $1,000 - $5,000: 0.1% commission")
print("    $5,000+:         0.05% commission")

# Calculate examples manually
print("\nExamples:")
print(f"  Trade $500:    ${500 * 0.002:.2f} (all in tier 1)")
print(f"  Trade $3,000:  ${1000 * 0.002 + 2000 * 0.001:.2f} (tier 1 + tier 2)")
print(f"  Trade $8,000:  ${1000 * 0.002 + 4000 * 0.001 + 3000 * 0.0005:.2f} (all tiers)")

In [ ]:
# Run backtest with tiered commission
backtest = Backtest(data=data, config=config_tiered)
results_tiered = backtest.run([strategy])

metrics = results_tiered[strategy.get_name()]["metrics"]
print("\nPerformance with Tiered Commission:")
print(f"  Total Return:  {metrics['total_return']:.2f}%")
print(f"  Total Trades:  {metrics['total_trades']}")
print(f"  Sharpe Ratio:  {metrics['sharpe_ratio']:.2f}")

## 4. Custom Commission Models

Create your own commission structure by inheriting from the `Commission` base class.

Let's create two deterministic custom models:
1. **Minimum Fee Commission**: Percentage with a minimum fee
2. **Capped Commission**: Percentage bounded by minimum and maximum fees

Callbacks passed to `Backtest` must be deterministic and side-effect free.

In [ ]:
class MinimumFeeCommission(Commission):
    """Percentage commission with a minimum fee.

    Common structure: Pay percentage or minimum fee, whichever is higher.
    Example: 0.1% or $5 minimum.
    """

    def __init__(self, rate: float, min_fee: float, name: str = None):
        super().__init__(name=name or f"MinFee({rate * 100:.2f}%, ${min_fee})")
        self.rate = rate
        self.min_fee = min_fee

    def calculate(self, shares: float, price: float) -> float:
        """Calculate commission as percentage with minimum fee."""
        trade_value = shares * price
        percentage_fee = trade_value * self.rate

        # Return whichever is higher: percentage or minimum
        return max(percentage_fee, self.min_fee)


print("✓ MinimumFeeCommission created")

In [ ]:
class CappedCommission(Commission):
    """Percentage commission bounded by minimum and maximum fees."""

    def __init__(
        self,
        rate: float = 0.002,
        min_fee: float = 1.0,
        max_fee: float = 20.0,
        name: str | None = None,
    ):
        super().__init__(name=name or f"Capped({rate * 100:.2f}%)")
        self.rate = rate
        self.min_fee = min_fee
        self.max_fee = max_fee

    def calculate(self, shares: float, price: float) -> float:
        """Calculate a deterministic bounded percentage fee."""
        return min(max(shares * price * self.rate, self.min_fee), self.max_fee)


print("✓ CappedCommission created")

### Test Custom Commissions

In [ ]:
# Test MinimumFeeCommission
min_fee_comm = MinimumFeeCommission(rate=0.001, min_fee=5.0)

print("MinimumFeeCommission Examples:")
print(f"  Trade $1,000:  ${min_fee_comm.calculate(10, 100):.2f} (min fee applies)")
print(f"  Trade $10,000: ${min_fee_comm.calculate(100, 100):.2f} (percentage applies)")
print(f"  Trade $20,000: ${min_fee_comm.calculate(200, 100):.2f} (percentage applies)")

In [ ]:
# Test CappedCommission
capped_comm = CappedCommission(rate=0.002, min_fee=1.0, max_fee=20.0)

print("\nCappedCommission Examples:")
for trade_value in [100, 1_000, 100_000]:
    fee = capped_comm.calculate(1, trade_value)
    print(f"  Trade ${trade_value:>7,}: ${fee:.2f}")

## 5. Commission Impact Analysis

Compare how different commission structures affect the same strategy.

In [ ]:
# Create configs with different commission structures
configs = {
    "Zero Commission": BacktestConfig(
        initial_capital=10000.0,
        lookback_period=50,
        commission_type="flat",
        commission_value=0.0,
        risk_free_rate=0.02,
    ),
    "Low (0.05%)": BacktestConfig(
        initial_capital=10000.0,
        lookback_period=50,
        commission_type="percentage",
        commission_value=0.0005,
        risk_free_rate=0.02,
    ),
    "Medium (0.1%)": BacktestConfig(
        initial_capital=10000.0,
        lookback_period=50,
        commission_type="percentage",
        commission_value=0.001,
        risk_free_rate=0.02,
    ),
    "High (0.25%)": BacktestConfig(
        initial_capital=10000.0,
        lookback_period=50,
        commission_type="percentage",
        commission_value=0.0025,
        risk_free_rate=0.02,
    ),
    "Flat $5": BacktestConfig(
        initial_capital=10000.0,
        lookback_period=50,
        commission_type="flat",
        commission_value=5.0,
        risk_free_rate=0.02,
    ),
}

print("Testing commission impact on MA strategy...")
print(f"Strategy: {strategy.get_name()}\n")

In [ ]:
# Run backtests with different commissions
comparison_results = []

for name, config in configs.items():
    backtest = Backtest(data=data, config=config)
    results = backtest.run([strategy])

    metrics = results[strategy.get_name()]["metrics"]

    comparison_results.append(
        {
            "Commission Type": name,
            "Total Return (%)": metrics["total_return"],
            "CAGR (%)": metrics["cagr"],
            "Sharpe Ratio": metrics["sharpe_ratio"],
            "Max Drawdown (%)": metrics["max_drawdown"],
            "Total Trades": metrics["total_trades"],
            "Win Rate (%)": metrics["win_rate"],
        }
    )

# Create comparison DataFrame
comparison_df = pd.DataFrame(comparison_results)

print("\nCOMMISSION IMPACT ANALYSIS")
print("=" * 100)
print(comparison_df.to_string(index=False))
print("=" * 100)

In [ ]:
# Calculate commission cost impact
zero_return = comparison_df[comparison_df["Commission Type"] == "Zero Commission"][
    "Total Return (%)"
].values[0]

print("\nCOMMISSION COST IMPACT:")
print("=" * 60)
for _, row in comparison_df.iterrows():
    if row["Commission Type"] != "Zero Commission":
        impact = zero_return - row["Total Return (%)"]
        impact_pct = (impact / zero_return * 100) if zero_return != 0 else 0
        print(
            f"{row['Commission Type']:.<20} Cost: {impact:>6.2f}% ({impact_pct:>5.1f}% of returns)"
        )
print("=" * 60)

## High-Frequency vs Low-Frequency Trading

Commission impact varies dramatically based on trading frequency!

In [ ]:
# Compare high-frequency (short MA windows) vs low-frequency (long MA windows)
from simple_backtest.strategy import BuyAndHoldStrategy

strategies_freq = [
    MovingAverageStrategy(short_window=5, long_window=15, shares=10, name="MA_High_Freq"),
    MovingAverageStrategy(short_window=20, long_window=50, shares=10, name="MA_Low_Freq"),
    BuyAndHoldStrategy(shares=50, name="Buy_Hold"),
]

# Test with high commission (0.25%)
config_high_comm = BacktestConfig(
    initial_capital=10000.0,
    lookback_period=60,
    commission_type="percentage",
    commission_value=0.0025,  # 0.25%
    risk_free_rate=0.02,
)

backtest = Backtest(data=data, config=config_high_comm)
results_freq = backtest.run(strategies_freq)

print("\nTRADING FREQUENCY vs COMMISSION IMPACT (0.25% commission)")
print("=" * 80)
for strat in strategies_freq:
    metrics = results_freq[strat.get_name()]["metrics"]
    print(f"\n{strat.get_name()}:")
    print(f"  Total Trades:  {metrics['total_trades']:>4}")
    print(f"  Total Return:  {metrics['total_return']:>7.2f}%")
    print(f"  Sharpe Ratio:  {metrics['sharpe_ratio']:>7.2f}")
print("=" * 80)

## Summary

In this notebook, we explored commission models:

1. ✅ **Percentage Commission**: Most common, scales with trade size
2. ✅ **Flat Commission**: Fixed fee, benefits larger trades
3. ✅ **Tiered Commission**: Volume discounts for large trades
4. ✅ **Custom Commissions**: Created MinimumFee and Progressive models
5. ✅ **Impact Analysis**: Measured how commissions affect performance

### Key Insights:

**Commission Impact:**
- Higher trading frequency = higher commission costs
- Even "low" commissions (0.1%) can significantly reduce returns
- High-frequency strategies need ultra-low commissions to be profitable
- Buy-and-hold strategies are less affected by commission rates

**Choosing Commission Type:**
- **Percentage**: Good for variable trade sizes
- **Flat**: Better for consistent large trades
- **Tiered**: Best for high-volume trading
- **Custom**: Match your exact broker structure

### Important Considerations:

1. **Don't Forget Commissions**: Always include realistic commission costs
2. **Match Your Broker**: Use the exact commission structure of your broker
3. **Trading Frequency**: High-frequency strategies are extremely sensitive to commissions
4. **Slippage**: Real costs include slippage (not just commission)
5. **Tax Impact**: Consider tax implications (not modeled here)

### Creating Custom Commissions:

To create your own commission model:
```python
from simple_backtest.commission import Commission

class MyCommission(Commission):
    def calculate(self, shares: float, price: float) -> float:
        # Your logic here
        return commission_amount
```

### Real-World Commission Examples:

**US Stock Brokers:**
- Robinhood, Webull: $0
- Interactive Brokers: $0.0035/share (min $0.35)
- TD Ameritrade: $0

**Crypto Exchanges:**
- Coinbase Pro: 0.5% (retail) to 0.04% (high volume)
- Binance: 0.1% (can be reduced with BNB)
- Kraken: 0.16% to 0.26%

**Forex Brokers:**
- Spread-based (2-5 pips)
- Commission: $3-$7 per lot

### Next Steps:

- Add slippage modeling to commissions
- Model market impact for large orders
- Include tax-aware backtesting
- Test strategies across different commission environments